In [1]:
import json
import numpy as np
import xobjects as xo
import xtrack as xt
import xpart as xp
from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy import constants 
####################
# Choose a context #
####################
context = xo.ContextCpu(omp_num_threads=4)
# context = xo.ContextCpu(omp_num_threads='auto')
buf = context.new_buffer()

#references: https://www.sciencedirect.com/science/article/pii/S0168900222011445?ref=pdf_download&fr=RR-2&rr=80ca25af8a5ace93

# Ion properties:
m_u = 931.49410242e6 # eV/c^2 -- atomic mass unit
A = 16 # Weight of O
#Z = 6  # Number of protons in the ion (O)
#m_e = 0.511e6 # eV/c^2 -- electron mass
m_p = 938.272088e6 # eV/c^2 -- proton mass
clight = 299792458.0 # m/s

q0=5

mass0 = A*m_u #+ Ne*m_e # eV/c^2

beta_rel = 0.64
gamma_rel = 1.30

p0c = mass0*beta_rel*gamma_rel #eV/c

# equiv_proton_momentum = 236e9 # eV/c = gamma_p*m_p*v
# gamma_p = np.sqrt( 1 + (equiv_proton_momentum/m_p)**2 ) # equvalent gamma for protons in the ring

# p0c = equiv_proton_momentum*(Z-Ne) # eV/c
gamma2 = np.sqrt( 1 + (p0c/mass0)**2 ) # ion relativistic factor
beta2 = np.sqrt(1-1/(gamma2*gamma2)) # ion beta

print(gamma2)
print(beta2)

particle_ref=xp.Particles(q0=q0,mass0=mass0,p0c=p0c)


1.3008551033839242
0.639579302749178


In [2]:
p0c

12400049491.415041

In [3]:
circumference =  128.80 #m
T = circumference/(clight*beta_rel)
s_per_turn = T
slip_factor=0.45

beta_x = 6
beta_y = 2

disp_x = 0
Q_x = 2.2
Q_y = 2.4
dQx = 0
dQy = 0

arc = xt.LineSegmentMap(
        
        )

line = xt.Line(
        elements=[arc])
line.particle_ref=particle_ref

qs=0.005247746218929317
bets0=-2078.673352643715

arc_matching = xt.LineSegmentMap(
        qx=Q_x, qy=Q_y,
        dqx=dQx, dqy=dQy,
        length=circumference,
        betx=beta_x,
        bety=beta_y,
        qs=qs,
        bets=bets0)

line_matching=xt.Line([arc_matching])
line_matching.particle_ref=particle_ref
line_matching.build_tracker()

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.


In [4]:
s_per_turn

6.71297741586281e-07

In [5]:
emittance_x=10*1e-6 #inital emittance
emittance_y=15*1e-6 #inital emittance
num_particles = int(1e4)

sigma_x = np.sqrt(beta_x*emittance_x)
sigma_px = np.sqrt(emittance_x*1/beta_x)
sigma_y = np.sqrt(beta_y*emittance_y)
sigma_py = np.sqrt(emittance_y*1/beta_y)
# sigma_p = 2e-5 # relative ion momentum spread
# sigma_p = 2e-4 # relative ion momentum spread

delta = np.linspace(-9.2e-5, 9.2e-5, num_particles)


# delta = np.random.normal(loc=0, scale=sigma_p, size=num_particles)
x = np.random.normal(loc=0.0, scale=sigma_x, size=num_particles) + disp_x * delta
px = np.random.normal(loc=0.0, scale=sigma_px, size=num_particles)
y = np.random.normal(loc=0.0, scale=sigma_y, size=num_particles)
py = np.random.normal(loc=0.0, scale=sigma_py, size=num_particles)

particles = xp.Particles(
    mass0=mass0,
    total_intensity_particles=int(1e8),
    p0c=p0c,
    q0=q0,
    x=x,
    px=px,
    y=y,
    py=py,
    delta=delta,
    zeta=0
)

print('sigma_px',sigma_px*1e3)



sigma_px 1.2909944487358056


In [6]:
gamma0=gamma_rel
beta0=beta_rel
gemitt_x = 14e-6
gemitt_y = 14e-6
# gemitt_x = 1e-12
# gemitt_y = 1e-12
gemitt_x = 0.6381603736404441*1e-6
gemitt_y = 0.6381603736404441*1e-6
nemitt_x = gemitt_x*beta0*gamma0
nemitt_y = gemitt_y*beta0*gamma0

bunch_intensity = int(1e8)


particles = xp.generate_matched_gaussian_bunch(
        num_particles=num_particles,
        total_intensity_particles=bunch_intensity,
        nemitt_x=nemitt_x, nemitt_y=nemitt_y, sigma_z=4.2,
        particle_ref=particle_ref,
        line=line_matching,        
        )

# create desired beam
#particles.delta = np.random.uniform(-sigma_dp,sigma_dp,num_particles)
particles.delta = np.linspace(-9.2e-5, 9.2e-5, num_particles)
particles.zeta = np.random.uniform(-circumference/2, circumference/2, num_particles)

In [13]:
tw=line_matching.twiss(method='4d')
sigma_delta = np.std(particles.delta)

bm_growth_rates = tw.get_ibs_growth_rates(
    formalism="bjorken-mtingwa",
    total_beam_intensity=bunch_intensity,
    nemitt_x=nemitt_x,
    nemitt_y=nemitt_y,
    sigma_delta=sigma_delta,
    bunch_length=circumference,
    bunched=False,
)

nag_growth_rates = tw.get_ibs_growth_rates(
    formalism="nagaitsev",
    total_beam_intensity=bunch_intensity,
    nemitt_x=nemitt_x,
    nemitt_y=nemitt_y,
    sigma_delta=sigma_delta,
    bunch_length=circumference,
    bunched=False,
)

/home/pkruyt/cernbox/xsuite-laser2024/xfields/xfields/ibs/_analytical.py:906: RuntimeWarning: invalid value encountered in divide
  - (betx * Hy) / (Hx * gemitt_y)
/home/pkruyt/cernbox/xsuite-laser2024/xfields/xfields/ibs/_analytical.py:907: RuntimeWarning: divide by zero encountered in divide
  + (betx / (Hx * gamma**2)) * (2 * betx_over_epsx - bety_over_epsy - gamma**2 / sigd**2)
/home/pkruyt/cernbox/xsuite-laser2024/xfields/xfields/ibs/_analytical.py:910: RuntimeWarning: divide by zero encountered in divide
  + (betx / (Hx * gamma**2)) * (6 * betx_over_epsx * gamma**2 * phix**2)
/home/pkruyt/cernbox/xsuite-laser2024/xfields/xfields/ibs/_analytical.py:910: RuntimeWarning: invalid value encountered in multiply
  + (betx / (Hx * gamma**2)) * (6 * betx_over_epsx * gamma**2 * phix**2)
/home/pkruyt/cernbox/xsuite-laser2024/xfields/xfields/ibs/_analytical.py:967: RuntimeWarning: divide by zero encountered in divide
  + (betx / (Hx * gamma**2))
/home/pkruyt/cernbox/xsuite-laser2024/xfields/

In [14]:
nag_growth_rates.Tz

0.0047606498665125315

In [15]:
bm_growth_rates.Tz

0.005371818130940169